In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly
import plotly.express as px

In [8]:
data = pd.read_csv("../data/processed/train.csv")
data_test = pd.read_csv("../data/processed/test.csv")

In [11]:
datasets_tt = {
    "train": data,
    "test": data_test,
}

# ===== LENGTH =====
for name, df in datasets_tt.items():
    df["char_len"] = df["text"].astype(str).apply(len)
    df["word_len"] = df["text"].astype(str).apply(lambda x: len(x.split()))

    print(f"\n{name.upper()} - WORD LEN")
    print(df["word_len"].describe())

# ===== CLASS =====
for name, df in datasets_tt.items():
    print(f"\n{name.upper()} - CLASS")
    print(df["label"].value_counts(normalize=True))

# ===== LEXICAL =====
def compute_lex(df):
    texts = df["text"].astype(str).tolist()
    tokens = [t.lower().split() for t in texts]
    all_tokens = [w for t in tokens for w in t]

    d1 = len(set(all_tokens)) / len(all_tokens)

    bigrams = []
    for t in tokens:
        bigrams.extend(list(zip(t, t[1:])))

    d2 = len(set(bigrams)) / len(bigrams) if len(bigrams) > 0 else 0

    return d1, d2

for name, df in datasets_tt.items():
    d1, d2 = compute_lex(df)
    print(f"\n{name.upper()} - LEX")
    print(f"distinct_1: {d1:.3f}, distinct_2: {d2:.3f}")

# ===== PLOT =====
plot_df = []

for name, df in datasets_tt.items():
    tmp = df.copy()
    tmp["source"] = name
    plot_df.append(tmp)

plot_df = pd.concat(plot_df)

px.histogram(
    plot_df,
    x="word_len",
    color="source",
    barmode="overlay",
    opacity=0.5,
    nbins=50,
    title="Train vs Test - Word Length"
).show()


TRAIN - WORD LEN
count    4649.000000
mean       14.118520
std        18.362749
min         1.000000
25%         4.000000
50%         8.000000
75%        15.000000
max       135.000000
Name: word_len, dtype: float64

TEST - WORD LEN
count    2214.000000
mean       11.904697
std        16.087718
min         1.000000
25%         4.000000
50%         7.000000
75%        13.000000
max       148.000000
Name: word_len, dtype: float64

TRAIN - CLASS
label
neutral     0.512153
positive    0.253818
negative    0.234029
Name: proportion, dtype: float64

TEST - CLASS
label
neutral     0.641373
positive    0.242096
negative    0.116531
Name: proportion, dtype: float64

TRAIN - LEX
distinct_1: 0.394, distinct_2: 0.854

TEST - LEX
distinct_1: 0.481, distinct_2: 0.906


### TRAIN vs TEST - Findings

- Length distribution is similar between train and test:
  - Both are right-skewed with long-tail behavior  
  - Comparable statistics (mean ≈ 14 vs 12, similar std and range)  
  - Both contain very short and very long texts  

- Class distribution differs:
  - Train:
    - Neutral ≈ 51%  
    - Positive ≈ 25%  
    - Negative ≈ 23%  
  - Test:
    - Neutral ≈ 64%  
    - Positive ≈ 24%  
    - Negative ≈ 12%  

- Lexical diversity is slightly higher in test data:
  - Distinct-1: 0.394 → 0.481  
  - Distinct-2: 0.854 → 0.906  

### Interpretation

- Structural properties (length, long-tail behavior) are consistent between train and test  

- However, test data exhibits:
  - stronger class imbalance (higher neutral, lower negative)  
  - higher lexical diversity  

- This suggests that:
  - train and test are generally aligned in structure  
  - but test data may be slightly more diverse and imbalanced  

- Despite these differences, the gap between train and test is significantly smaller than the gap between real and synthetic data  

- Therefore:
  - observed discrepancies between real and synthetic datasets cannot be explained by train/test variation  
  - they reflect genuine limitations of synthetic data generation  